# 05 — Ranked cases and locked holdout

Development cases are ranked by statistical evidence, not by population
size. Business priority remains separate because no validated criticality
data is available. Set `RUN_HOLDOUT=1` only once, after the design is frozen.


## 1. Setup


In [ ]:
import importlib
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

SECTOR = os.getenv("ANOMALY_SECTOR", "telecom")  # or "petrobras_3w"
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_10_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_10_1_run2",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json

CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
EVAL_ROOT = RUN_ROOT / "SPEC-EVAL"
SPLIT_ROOT = RUN_ROOT / "SPLITS"

import hashlib
import tempfile

import duckdb
import joblib

import evaluation_core
import simple_model_core
evaluation_core = importlib.reload(evaluation_core)
simple_model_core = importlib.reload(simple_model_core)
from evaluation_core import evaluate_cases, form_cases
from simple_model_core import (
    alerts_from_score_file, case_score_trace, materialize_measurement_features,
    materialize_wide_partition, partition_exposure, score_residual_file,
)

VERSION = "2.1.0"
MODEL_ROOT = DATA_ROOT / "outputs" / "models" / f"v{VERSION}" / SECTOR / f"{SECTOR}_models_v2_1_run1"
FINAL_ROOT = DATA_ROOT / "outputs" / "cases" / f"v{VERSION}" / SECTOR / f"{SECTOR}_cases_v2_1_run1"
EVALUATION_ROOT = DATA_ROOT / "outputs" / "evaluation" / f"v{VERSION}" / SECTOR / f"{SECTOR}_evaluation_v2_1_run1"
RUN_HOLDOUT = os.getenv("RUN_HOLDOUT", "0") == "1"

configuration = read_json(MODEL_ROOT / "selected_configuration.json")
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
bundle = joblib.load(MODEL_ROOT / "residual_bundle.joblib")
group_path = SPLIT_ROOT / "entity_groups.parquet"
entity_groups = pd.read_parquet(group_path) if group_path.is_file() else pd.DataFrame()
PARTITION = "holdout" if RUN_HOLDOUT else "development"
DEPLOYABLE = configuration["selection_status"] == "within_budget"
if RUN_HOLDOUT and not DEPLOYABLE:
    raise RuntimeError(
        "Holdout remains sealed: no development configuration met the "
        "predeclared case-workload budget"
    )

display(pd.Series({
    "partition": PARTITION,
    "portfolio": configuration["portfolio"],
    "development_status": configuration["selection_status"],
    "holdout_opened": RUN_HOLDOUT,
    "output": str(FINAL_ROOT),
}, name="value").to_frame())


## 2. Load development cases or score the one-time holdout


In [ ]:
if not RUN_HOLDOUT:
    alerts = pd.read_parquet(MODEL_ROOT / "selected_alerts.parquet")
    cases = pd.read_parquet(MODEL_ROOT / "selected_cases.parquet")
    members = pd.read_parquet(MODEL_ROOT / "selected_case_members.parquet")
    metrics = pd.read_csv(MODEL_ROOT / "development_metrics.csv")
    fault_types = pd.read_csv(MODEL_ROOT / "fault_type_results.csv")
    case_trace = pd.read_parquet(MODEL_ROOT / "development_case_trace.parquet")
else:
    receipt = FINAL_ROOT.parent / "HOLDOUT_USED.json"
    if receipt.exists():
        raise RuntimeError(f"Holdout already used. Receipt: {receipt}")
    truth_root = EVALUATION_ROOT / "holdout_sealed"
    fault_events = pd.read_parquet(truth_root / "fault_events.parquet")
    fault_intervals = pd.read_parquet(truth_root / "fault_entity_intervals.parquet")
    decisions = configuration["feature_settings"]

    temporary = tempfile.TemporaryDirectory()
    work = Path(temporary.name)
    wide, features, scores = work / "wide.parquet", work / "features.parquet", work / "scores.parquet"
    split = materialize_wide_partition(
        CORE_ROOT, SPLIT_ROOT, "holdout", catalogue, wide,
        lookback_seconds=configuration["lookback_seconds"],
    )
    materialize_measurement_features(wide, catalogue, features)
    score_residual_file(
        bundle, features, scores,
        cadence_seconds=decisions["base_cadence_seconds"],
        dispersion_window_seconds=decisions["dispersion_window_seconds"],
        cusum_allowance=configuration["cusum_allowance"],
        score_start=split["score_start"], score_end=split["score_end"],
    )
    frames = []
    for channel in configuration["channels"]:
        frame = alerts_from_score_file(
            scores, channel, configuration["thresholds"][channel],
            min_consecutive=configuration["channel_persistence"][channel],
            recovery_consecutive=configuration["recovery_observations"],
        )
        if not frame.empty:
            frames.append(frame)
    if frames:
        alerts = pd.concat(frames, ignore_index=True).sort_values("alert_start").reset_index(drop=True)
        alerts["alert_id"] = [f"A-{number:09d}" for number in range(1, len(alerts) + 1)]
    else:
        from evaluation_core import ALERT_COLUMNS
        alerts = pd.DataFrame(columns=ALERT_COLUMNS)
    cases, members = form_cases(
        alerts, entity_groups,
        gap_seconds=configuration["case_gap_seconds"],
        thresholds=configuration["thresholds"],
    )
    exposure = partition_exposure(
        scores, configuration["exposure_unit"], decisions["base_cadence_seconds"]
    )
    result = evaluate_cases(
        cases, members, fault_events, fault_intervals,
        exposure_value=exposure,
        exposure_unit=configuration["exposure_unit"],
        decision_horizon_seconds=configuration["decision_horizon_seconds"],
    )
    metrics, fault_types = result["metrics"], result["fault_type_results"]
    case_trace = case_score_trace(
        scores, features, cases, members, alerts, bundle,
        configuration["thresholds"], configuration["channels"],
        window_seconds=max(
            decisions["dispersion_window_seconds"],
            20 * decisions["base_cadence_seconds"],
        ),
    )


## 3. Rank evidence and keep operational priority separate


In [ ]:
cases = cases.sort_values(
    ["anomaly_evidence_score", "case_start"], ascending=[False, True]
).reset_index(drop=True)
if "rank" in cases:
    cases = cases.drop(columns="rank")
cases.insert(0, "rank", np.arange(1, len(cases) + 1))
cases["operational_priority"] = pd.NA
cases["priority_status"] = "not_scored_no_validated_criticality_data"

display(cases.head(25))
display(metrics)
display(fault_types)


## 4. Save the product-facing evidence


In [ ]:
with new_output_directory(FINAL_ROOT) as output:
    cases.to_csv(output / "ranked_cases.csv", index=False)
    alerts.to_parquet(output / "alerts.parquet", index=False)
    members.to_parquet(output / "case_members.parquet", index=False)
    metrics.to_csv(output / "evaluation_metrics.csv", index=False)
    fault_types.to_csv(output / "fault_type_results.csv", index=False)
    case_trace.to_parquet(output / "case_score_trace.parquet", index=False)
    write_json(output / "case_run.json", {
        "version": VERSION,
        "sector": SECTOR,
        "partition": PARTITION,
        "development_status": configuration["selection_status"],
        "configuration_sha256": hashlib.sha256(
            (MODEL_ROOT / "selected_configuration.json").read_bytes()
        ).hexdigest(),
        "evidence_score": "maximum channel peak divided by its calibration threshold",
        "operational_priority": "not available without validated criticality data",
    })

if RUN_HOLDOUT:
    write_json(FINAL_ROOT.parent / "HOLDOUT_USED.json", {
        "sector": SECTOR, "configuration": str(MODEL_ROOT), "result": str(FINAL_ROOT)
    })
    temporary.cleanup()

print("Saved:", FINAL_ROOT)
print("Next: 06_PRODUCT_DEMO.ipynb")
